In [ ]:
from sagemaker.tuner import (
    IntegerParameter,
    ContinuousParameter,
    HyperparameterTuner
)
from sagemaker.sklearn.estimator import SKLearn

# Estimador base
estimator = SKLearn(
    entry_point="train.py",
    role="arn:aws:iam::<ACCOUNT_ID>:role/LabRole",
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    output_path="s3://pablo-proyectoentrega4/models/",
    base_job_name="alquiler-tuning",
    enable_sagemaker_metrics=True
)

# Rango de hiperparámetros a explorar
hyperparameter_ranges = {
    "n-estimators":  IntegerParameter(50, 200),
    "max-depth":     IntegerParameter(3, 8),
    "learning-rate": ContinuousParameter(0.01, 0.3)
}

# Configurar el tuner
tuner = HyperparameterTuner(
    estimator=estimator,
    objective_metric_name="r2",
    metric_definitions=[
        {"Name": "r2", "Regex": "R2:\\s+([0-9\\.]+)"}
    ],
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=4,
    max_parallel_jobs=2,
    strategy="Bayesian"
)

# Lanzar el tuning
tuner.fit({"train": "s3://pablo-proyectoentrega4/processed/"})
print("Tuning job lanzado.")

No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


.

.

.

...

.

.

.

!


Tuning job lanzado.


In [ ]:
import sagemaker
from sagemaker.clarify import SageMakerClarifyProcessor, DataConfig, BiasConfig

session = sagemaker.Session()

clarify_processor = SageMakerClarifyProcessor(
    role="arn:aws:iam::<ACCOUNT_ID>:role/LabRole",
    instance_count=1,
    instance_type="ml.m5.large",
    sagemaker_session=session
)

data_config = DataConfig(
    s3_data_input_path="s3://pablo-proyectoentrega4/processed/",
    s3_output_path="s3://pablo-proyectoentrega4/clarify-output",
    label="price",
    dataset_type="text/csv"
)

bias_config = BiasConfig(
    label_values_or_threshold=[500],
    facet_name="district"
)

clarify_processor.run_pre_training_bias(
    data_config=data_config,
    data_bias_config=bias_config,
    methods="all",
    wait=True,
    logs=True
)

print("Análisis de sesgo completado.")

INFO:sagemaker.image_uris:Defaulting to the only supported framework/algorithm version: 1.0.


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


INFO:sagemaker.clarify:Analysis Config: {'dataset_type': 'text/csv', 'label': 'price', 'label_values_or_threshold': [500], 'facet': [{'name_or_index': 'district'}], 'methods': {'report': {'name': 'report', 'title': 'Analysis Report'}, 'pre_training_bias': {'methods': 'all'}}}


INFO:sagemaker:Creating processing-job with name Clarify-Pretraining-Bias-2026-06-10-14-48-36-166


.

...

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml
We are not in a supported iso region, /bin/sh exiting gracefully with no changes.
INFO:sagemaker-clarify-processing:Starting SageMaker Clarify Processing job
INFO:analyzer.data_loading.data_loader_util:Analysis config path: /opt/ml/processing/input/config/analysis_config.json
INFO:analyzer.data_loading.data_loader_util:Analysis result path: /opt/ml/processing/output
INFO:analyzer.data_loading.data_loader_util:This host is algo-1.
INFO:analyzer.data_loading.data_loader_util:This host is the leader.
INFO:analyzer.data_loading.data_loader_util:Number of hosts in the cluster is 1.
INFO:sagemaker-clarify-processing:Running Python / Pandas based analyzer.
INFO:analyzer.data_loading.data_loader_factory:Dataset type: text/csv uri: /opt/ml/processing/input/data
INFO:sagemaker-clarify-processing:Loadin

#015[Stage 0:>                                                          (0 + 2) / 2]#015#015                                                                                #015#015[Stage 3:>                                                          (0 + 2) / 2]#015#015                                                                                #015INFO:analyzer.utils.spark_util:Report Metadata: Could not find predicted_label in Dataframe
INFO:sagemaker-clarify-processing:Calculated global analysis without predictor
INFO:sagemaker-clarify-processing:Collected analyses: 
{'version': '1.0', 'pre_training_bias_metrics': {'label': 'price', 'facets': {'district': [{'value_or_threshold': 'Chamartín', 'metrics': [{'name': 'CDDL', 'description': 'Conditional Demographic Disparity in Labels (CDDL)', 'value': None, 'error': 'Group variable is empty or not provided'}, {'name': 'CI', 'description': 'Class Imbalance (CI)', 'value': 0.8392025138151479}, {'name': 'DPL', 'description': 'Difference in

INFO:analyzer.utils.util:['jupyter', 'nbconvert', '--to', 'html', '--output', '/opt/ml/processing/output/report.html', '/opt/ml/processing/output/report.ipynb', '--template', 'sagemaker-xai']
[NbConvertApp] Converting notebook /opt/ml/processing/output/report.ipynb to html
[NbConvertApp] Writing 4669413 bytes to /opt/ml/processing/output/report.html
INFO:analyzer.utils.util:['wkhtmltopdf', '-q', '--enable-local-file-access', '/opt/ml/processing/output/report.html', '/opt/ml/processing/output/report.pdf']


INFO:analyzer.utils.system_util:exit_message: Completed: SageMaker XAI Analyzer ran successfully
INFO:py4j.clientserver:Closing down clientserver connection


Análisis de sesgo completado.
